In [1]:
import pandas as pd
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit

# 1. Load your dataset
df = pd.read_csv("../preprocessing/train_data_with_preprocessing2.csv")

# 2. Build the Bipartite Graph
G = nx.Graph()

# Add prefixes to avoid collisions between identical context and reply strings
df['ctx_node'] = "CTX_" + df['prev_clean'].astype(str)
df['rep_node'] = "REP_" + df['target_clean'].astype(str)

# Create edges based on rows
edges = list(zip(df['ctx_node'], df['rep_node']))
G.add_edges_from(edges)

# 3. Find Connected Components
components = list(nx.connected_components(G))

# Map every node to a unique Component ID
node_to_component = {}
for component_id, nodes in enumerate(components):
    for node in nodes:
        node_to_component[node] = component_id

# 4. Map the Component IDs back to the original dataframe
# (You only need to check the context node, as the reply node is guaranteed to have the same ID)
df['component_id'] = df['ctx_node'].map(node_to_component)

# 5. Split the data ensuring components are kept together
# GroupShuffleSplit ensures all rows with the same component_id stay in the same split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(gss.split(df, groups=df['component_id']))

train_df = df.iloc[train_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])
val_df = df.iloc[val_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")

Training rows: 17991
Validation rows: 1440


In [2]:
train_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)
val_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
  accuracy_score,
  f1_score,
  classification_report,
  confusion_matrix
)

In [3]:
target_col = 'label'

In [4]:
# Extract text columns for TF-IDF
X_context_train = train_df['context'].fillna("").astype(str)
X_reply_train = train_df['reply'].fillna("").astype(str)
y_train = train_df[target_col].fillna("nan").astype(str)

X_context_dev = val_df['context'].fillna("").astype(str)
X_reply_dev = val_df['reply'].fillna("").astype(str)
y_dev = val_df[target_col].fillna("nan").astype(str)

In [5]:
# Word-level TF-IDF
context_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [6]:
# Character-level TF-IDF
context_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [7]:
# Fit only on training data, transform both
Cw_train = context_word.fit_transform(X_context_train)
Cw_dev = context_word.transform(X_context_dev)

Rw_train = reply_word.fit_transform(X_reply_train)
Rw_dev = reply_word.transform(X_reply_dev)

Cc_train = context_char.fit_transform(X_context_train)
Cc_dev = context_char.transform(X_context_dev)

Rc_train = reply_char.fit_transform(X_reply_train)
Rc_dev = reply_char.transform(X_reply_dev)

In [8]:
# Combine all four sparse feature matrices
X_train = hstack([Cw_train, Cc_train, Rw_train, Rc_train], format="csr")
X_dev = hstack([Cw_dev, Cc_dev, Rw_dev, Rc_dev], format="csr")

print("Training feature matrix:", X_train.shape)
print("Development feature matrix:", X_dev.shape)

Training feature matrix: (17991, 36109)
Development feature matrix: (1440, 36109)


In [9]:
import optuna
import numpy as np
from sklearn.metrics import log_loss

optuna.logging.set_verbosity(optuna.logging.DEBUG)

def objective(trial):
  n_estimators = trial.suggest_int("n_estimators", 50, 300)
  max_depth = trial.suggest_int("max_depth", 3, 32)
  min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
  min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
  max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", "none"])

  if max_features == "none":
    max_features = None

  criterion = trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"])
  bootstrap = trial.suggest_categorical("bootstrap", [True, False])

  # 6
  model = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    max_features=max_features,
    criterion=criterion,
    bootstrap=bootstrap,
    random_state=42
  )

  model.fit(X_train, y_train)
  # y_pred = model.predict(X_dev)
  # score = accuracy_score(y_dev, y_pred) 
  y_proba = model.predict_proba(X_dev)
  score = log_loss(y_dev, y_proba)  # Use log loss for evaluation

  return score

In [10]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))

[I 2026-11-01 16:21:33,627] A new study created in memory with name: no-name-5374e1d6-2c54-4f96-9e13-c8d1a91f28b7


In [ ]:
study.optimize(objective, n_trials=50, gc_after_trial=True, n_jobs=1)

[I 2026-11-01 16:34:35,490] Trial 18 finished with value: 2.9616571988386284 and parameters: {'n_estimators': 121, 'max_depth': 27, 'min_samples_split': 6, 'min_samples_leaf': 16, 'max_features': 'log2', 'criterion': 'gini', 'bootstrap': False}. Best is trial 0 with value: 2.12771828325715.
[I 2026-11-01 16:34:35,890] Trial 20 finished with value: 2.992701097090484 and parameters: {'n_estimators': 104, 'max_depth': 23, 'min_samples_split': 14, 'min_samples_leaf': 8, 'max_features': 'log2', 'criterion': 'log_loss', 'bootstrap': True}. Best is trial 0 with value: 2.12771828325715.
[I 2026-11-01 16:34:36,379] Trial 22 finished with value: 3.1448350310596855 and parameters: {'n_estimators': 187, 'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 4, 'max_features': 'log2', 'criterion': 'gini', 'bootstrap': True}. Best is trial 0 with value: 2.12771828325715.
[I 2026-11-01 16:34:37,174] Trial 13 finished with value: 3.0455188123694192 and parameters: {'n_estimators': 231, 'max_dept

# Added evaluation: binary toxicity and 12-class subtype

New completed baseline: the original notebook contains a tuning study but no final selected model or test evaluation. This 100-tree baseline is not presented as the result of that unfinished search.

The original cells and outputs above are unchanged. **Run this added section independently.** It reads the supplied preprocessing2 CSV pair and preserves the literal `None` label. Every model in this comparison fits its word/character TF-IDF vectorizers on **all 19,431 training rows**, then evaluates the same **1,804 test rows**. The feature settings and block order match the original hyperparameter notebooks. Separate classifiers learn binary `is_toxic` and 12-class `label_id` targets.

This is a fresh full-training refit, so these test scores can differ from older outputs based on a smaller training partition. No new hyperparameter search uses the test set. Rare-class performance requires macro F1 as well as accuracy. The test has already been examined during the course project, so these comparisons are exploratory.

Dependencies: `numpy pandas scipy scikit-learn`.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from IPython.display import display

for dual_folder in [Path.cwd(), *Path.cwd().parents]:
    if (dual_folder/'preprocessing/train_data_with_preprocessing2.csv').exists():
        dual_root=dual_folder
        break
else:
    raise FileNotFoundError('Open this notebook inside the project folder.')
dual_train_path=dual_root/'preprocessing/train_data_with_preprocessing2.csv'
dual_test_path=dual_root/'preprocessing/test_data_with_preprocessing2.csv'
dual_signature={path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in [dual_train_path,dual_test_path]}

# Reuse fixed features when several model notebooks run in the same kernel.
if globals().get('dual_cached_signature') != dual_signature:
    dual_train=pd.read_csv(dual_train_path,keep_default_na=False)
    dual_test=pd.read_csv(dual_test_path,keep_default_na=False)
    dual_names=dict(zip(dual_train.label_id,dual_train.label))
    assert sorted(dual_names)==list(range(12)) and dual_names[0]=='None'
    for dual_frame in [dual_train,dual_test]:
        assert (dual_frame.label_id.map(dual_names)==dual_frame.label).all()
        assert ((dual_frame.label=='None')==(dual_frame.is_toxic==0)).all()
    dual_vectorizers=[]
    dual_train_blocks=[]
    dual_test_blocks=[]
    for train_column,test_column in [('prev_clean','context_clean'),('target_clean','reply_clean')]:
        for analyzer,ngrams in [('word',(1,2)),('char_wb',(3,5))]:
            vectorizer=TfidfVectorizer(analyzer=analyzer,ngram_range=ngrams,min_df=2,max_features=50000,sublinear_tf=True)
            dual_train_blocks.append(vectorizer.fit_transform(dual_train[train_column]))
            dual_test_blocks.append(vectorizer.transform(dual_test[test_column]))
            dual_vectorizers.append(vectorizer)
    dual_X_train=hstack(dual_train_blocks,format='csr')
    dual_X_test=hstack(dual_test_blocks,format='csr')
    dual_cached_signature=dual_signature.copy()
print('Train / test rows:',len(dual_train),len(dual_test))
print('TF-IDF features:',dual_X_train.shape,dual_X_test.shape)


Train / test rows: 19431 1804
TF-IDF features: (19431, 38806) (1804, 38806)


In [2]:
def dual_make_model():
    return RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=2)

dual_rows=[]
dual_predictions={}
dual_reports={}
dual_models={}
for dual_task,dual_column,dual_labels,dual_label_names in [
    ('binary','is_toxic',[0,1],['Non-toxic','Toxic']),
    ('subtype','label_id',list(range(12)),[dual_names[i] for i in range(12)]),
]:
    dual_model=dual_make_model()
    print('Training Random Forest:',dual_task,flush=True)
    dual_y_train=dual_train[dual_column].to_numpy(dtype=np.int64)
    dual_y_test=dual_test[dual_column].to_numpy(dtype=np.int64)
    dual_model.fit(dual_X_train,dual_y_train)
    dual_pred=np.asarray(dual_model.predict(dual_X_test)).reshape(-1).astype(np.int64)
    dual_models[dual_task]=dual_model
    dual_predictions[dual_task]=dual_pred.tolist()
    dual_rows.append(dict(
        model='Random Forest',task=dual_task,train_rows=len(dual_train),test_rows=len(dual_test),
        accuracy=float(accuracy_score(dual_y_test,dual_pred)),
        macro_f1=float(f1_score(dual_y_test,dual_pred,labels=dual_labels,average='macro',zero_division=0)),
        weighted_f1=float(f1_score(dual_y_test,dual_pred,labels=dual_labels,average='weighted',zero_division=0)),
    ))
    dual_reports[dual_task]=classification_report(dual_y_test,dual_pred,labels=dual_labels,target_names=dual_label_names,zero_division=0,output_dict=True)
    print(classification_report(dual_y_test,dual_pred,labels=dual_labels,target_names=dual_label_names,zero_division=0))
    print('Metrics:',dual_rows[-1],flush=True)
dual_results=pd.DataFrame(dual_rows)
display(dual_results.round(4))


Training Random Forest: binary
              precision    recall  f1-score   support

   Non-toxic       0.52      0.09      0.15       786
       Toxic       0.57      0.94      0.71      1018

    accuracy                           0.57      1804
   macro avg       0.54      0.51      0.43      1804
weighted avg       0.55      0.57      0.46      1804

Metrics: {'model': 'Random Forest', 'task': 'binary', 'train_rows': 19431, 'test_rows': 1804, 'accuracy': 0.5665188470066519, 'macro_f1': 0.428720914222773, 'weighted_f1': 0.4648034428975584}
Training Random Forest: subtype
                                     precision    recall  f1-score   support

                               None       0.43      0.75      0.54       786
                     Dehumanization       0.13      0.10      0.11        20
                         Harassment       0.00      0.00      0.00        88
                        Hate Speech       0.00      0.00      0.00        26
                    Identity Att

,model,task,train_rows,test_rows,accuracy,macro_f1,weighted_f1
0,Random Forest,binary,19431,1804,0.5665,0.4287,0.4648
1,Random Forest,subtype,19431,1804,0.3703,0.0878,0.2873
